## `delex_lite` from Python

Use this notebook to build and run a blocking program in Python, inspect saved candidates, and evaluate the results. It reproduces the Part 5 program from the [tutorial](doc/tutorial.md) and compares several values of K. Complete Parts 1 to 6 of the tutorial first for an introduction to rules, filters and evaluation.

Select each code cell and press Shift+Enter to run it. Work through the cells in order and compare the output with the explanation above each cell. Run times may vary depending on your machine.

### 1. The smoke test

Run the next cell to repeat the installation check, which performs a small search. It prints `OK — delex_lite runs on this machine.`

In [ ]:
!delex-lite smoke

### 2. Writing a program

Run the next cell to build the program from Part 5 of the tutorial. Its keep rule finds similar titles with BM25 (K = 10), then requires author similarity of at least 0.5. Its drop rule excludes pairs with different publication years. `&` joins the predicates of a rule, and `~` inverts `exact`. The comparison is in parentheses, since `&` is applied before `>=`.

Inspect the printed plan: it identifies the source, the author filter and the year exclusion, corresponding to the `--check` output in Part 4.

In [ ]:
import delex_lite as D

A, B, G = ("data/dblp_acm/table_a.parquet",
           "data/dblp_acm/table_b.parquet",
           "data/dblp_acm/gold.parquet")

prog = D.program(
    keep=[D.bm25("title", "title").topk(10) & (D.cosine("authors", "authors") >= 0.5)],
    drop=[~D.exact("year", "year")],
)
print(prog.explain())

### 3. Running the program and measuring recall

Run the next cell to apply the program to the two `data/dblp_acm` tables and evaluate it against the gold set. `D.block` performs the same run as the command-line tool.

Inspect the funnel: 22,702 pairs are found by the title source, 3,072 pass the author filter, and the drop rule removes 690. The resulting 2,382 candidates retain 98.07% of the known matches, with measured precision of 91.56%.

In [ ]:
summary = D.block(prog, A, B, gold=G)

Run the next cell to access the summary as a dictionary. It prints the candidate count (2,382), reduction (99.9603%), recall (98.07%), precision (91.56%), and runtime in seconds.

In [ ]:
print("candidates:", f"{summary['n_candidates']:,}")
print("reduction: ", f"{summary['reduction']:.4%}")
print("recall:    ", f"{summary['recall']:.2%}")
print("precision: ", f"{summary['precision']:.2%}")
print("time:      ", f"{summary['seconds']:.2f} s")

### 4. Saving and reading the candidate pairs

Run the next cell to save candidates as `candidates.parquet` in the notebook folder and display the first 10 pairs. The `out` argument corresponds to the command-line `--out` option. The example reads the saved file with `pyarrow`, which is installed with `delex_lite`.

The saved file contains 2,382 pairs. The `id1` column identifies records from table A, and `id2` identifies records from table B. The preview shows one candidate each for records 0 through 9 in table B. In the first pair, record 0 from table B is paired with record 2123 from table A.

In [ ]:
import pyarrow.parquet as pq

D.block(prog, A, B, out="candidates.parquet", verbose=False)

pairs = pq.read_table("candidates.parquet")
print(f"{pairs.num_rows:,} candidate pairs\n")
print("  id1    id2")
for row in pairs.slice(0, 10).to_pylist():
    print(f"{row['id1']:>5}  {row['id2']:>5}")

### 5. Measuring the recall of saved candidates

Run the next cell to evaluate the saved candidates against the gold set without repeating blocking.

The report contains 2,382 candidates (`n_candidates`) and 2,224 known matches (`n_gold`). Of those known matches, 2,181 are found (`tp`) and 43 are missing (`fn`). Recall is approximately 0.9807 (98.07%) and precision is approximately 0.9156 (91.56%).

In [ ]:
report = D.evaluate("candidates.parquet", G)
print(report)

### 6. Choosing K

A smaller K gives fewer candidates and less work for the matcher, and a larger K keeps more of the true matches. Run the next cell to compare six values of K using the title-only program from Part 1. Compare the candidate count, recall and precision in the resulting table:

```
   K   candidates    recall  precision
   1        2,289    97.53%     94.76%
   2        4,571    99.06%     48.20%
   5       11,395    99.51%     19.42%
  10       22,702    99.78%      9.77%
  20       45,162   100.00%      4.92%
  50      111,366   100.00%      2.00%
```

On these tables the recall reaches 100% at K = 20, with 45,162 candidates. In this dataset, the filter and drop rule in cell 2 increase measured precision. On other data they can also exclude true matches, so compare both recall and the number of candidates.

In [ ]:
print(f"{'K':>4}  {'candidates':>11}  {'recall':>8}  {'precision':>9}")
for k in [1, 2, 5, 10, 20, 50]:
    p = D.program(keep=[D.bm25("title", "title").topk(k)])
    s = D.block(p, A, B, gold=G, verbose=False, progress=False)
    print(f"{k:>4}  {s['n_candidates']:>11,}  {s['recall']:>8.2%}  {s['precision']:>9.2%}")

### 7. The program as JSON

Run the next cell to export the program as JSON and load it again with `D.from_json`. The JSON describes the same rules as `programs/papers.json`. The final `True` confirms that the exported and reloaded programs have the same JSON representation.

In [ ]:
text = prog.to_json(as_string=True)
print(text)
print(D.from_json(text).to_json() == prog.to_json())

### Going further

- To use your own data, replace the `A` and `B` paths with your input files and update the program's column names. See [File formats](doc/technical-guide.md#file-formats) for supported files and Python objects.
- See [The predicates](doc/technical-guide.md#the-predicates) for comparison functions and parameters, and [How sources and filters are chosen](doc/technical-guide.md#how-sources-and-filters-are-chosen) for their roles within a rule.
- Before using your own tables, check [What your files need](doc/technical-guide.md#what-your-files-need), including how to prepare record IDs.